In [47]:
import pandas as pd

In [48]:
delivery = pd.read_csv('deliveries.csv')
player = pd.read_csv('player.csv')
player_captain = pd.read_csv('player_captain.csv')

In [49]:
delivery

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260915,1426312,2,Kolkata Knight Riders,Sunrisers Hyderabad,9,5,SS Iyer,AK Markram,VR Iyer,1,0,1,NaN,0,NaN,NaN,NaN
260916,1426312,2,Kolkata Knight Riders,Sunrisers Hyderabad,9,6,VR Iyer,AK Markram,SS Iyer,1,0,1,NaN,0,NaN,NaN,NaN
260917,1426312,2,Kolkata Knight Riders,Sunrisers Hyderabad,10,1,VR Iyer,Shahbaz Ahmed,SS Iyer,1,0,1,NaN,0,NaN,NaN,NaN
260918,1426312,2,Kolkata Knight Riders,Sunrisers Hyderabad,10,2,SS Iyer,Shahbaz Ahmed,VR Iyer,1,0,1,NaN,0,NaN,NaN,NaN


In [50]:
player

,Player_Id,Player_Name,DOB,Batting_Hand,Bowling_Skill,Country,Is_Umpire,Unnamed: 7
0,1,SC Ganguly,8-Jul-72,Left_Hand,Right-arm medium,India,0,NaN
1,2,BB McCullum,27-Sep-81,Right_Hand,Right-arm medium,New Zealand,0,NaN
2,3,RT Ponting,19-Dec-74,Right_Hand,Right-arm medium,Australia,0,NaN
3,4,DJ Hussey,15-Jul-77,Right_Hand,Right-arm offbreak,Australia,0,NaN
4,5,Mohammad Hafeez,17-Oct-80,Right_Hand,Right-arm offbreak,Pakistan,0,NaN
...,...,...,...,...,...,...,...,...
518,519,Subroto Das,NaN,NaN,NaN,India,1,NaN
519,520,K Srinivasan,NaN,NaN,NaN,India,1,NaN
520,521,VK Sharma,NaN,NaN,NaN,India,1,NaN
521,523,AV Wankhade,14-Mar-92,Right_Hand,NaN,India,0,NaN


In [51]:
player_captain

,Match_Id,Player_Id,Team_Id,Is_Keeper,Is_Captain
0,335987,1,1,0,1
1,335987,2,1,0,0
2,335987,3,1,0,0
3,335987,4,1,0,0
4,335987,5,1,0,0
...,...,...,...,...,...
12689,829762,401,2,0,0
12690,829762,311,2,0,0
12691,829762,378,2,0,0
12692,829762,140,2,0,0


In [52]:
temp_df = player.merge(player_captain, on='Player_Id')[['Player_Name','Match_Id','Is_Captain']]

In [53]:
temp_df.head(1)

,Player_Name,Match_Id,Is_Captain
0,SC Ganguly,335987,1


In [54]:
delivery = delivery.merge(temp_df , left_on=['match_id','batter'], right_on=['Match_Id','Player_Name'], how='left').fillna(0)

In [55]:
delivery.head(1)

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder,Player_Name,Match_Id,Is_Captain
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,0,0,0,0,0.0,0.0


In [56]:
runs = delivery.groupby(['match_id','batter'])['batsman_runs'].sum().reset_index()

In [57]:
balls = delivery.groupby(['match_id','batter'])['batsman_runs'].count().reset_index()

In [58]:
fours = delivery.query('batsman_runs == 4').groupby(['match_id','batter'])['batsman_runs'].count().reset_index()

In [59]:
sixes = delivery.query('batsman_runs == 6').groupby(['match_id','batter'])['batsman_runs'].count().reset_index()

In [60]:
final_df = runs.merge(balls, on=['match_id','batter'], suffixes=('_runs','_balls')).merge(fours, on=['match_id','batter'], how='left').merge(sixes, on=['match_id','batter'], how='left')

In [61]:
final_df.fillna(0, inplace=True)

In [62]:
final_df.rename(columns={
    'batsman_runs_runs':'runs',
    'batsman_runs_balls':'balls',
    'batsman_runs_x':'fours',
    'batsman_runs_y':'sixes',
}, inplace=True)

In [63]:
final_df.head()

,match_id,batter,runs,balls,fours,sixes
0,335982,AA Noffke,9,12,1.0,0.0
1,335982,B Akhil,0,2,0.0,0.0
2,335982,BB McCullum,158,77,10.0,13.0
3,335982,CL White,6,10,0.0,0.0
4,335982,DJ Hussey,12,12,1.0,0.0


In [64]:
final_df['sr'] = round((final_df['runs']/final_df['balls'])*100,2)

In [65]:
final_df

,match_id,batter,runs,balls,fours,sixes,sr
0,335982,AA Noffke,9,12,1.0,0.0,75.00
1,335982,B Akhil,0,2,0.0,0.0,0.00
2,335982,BB McCullum,158,77,10.0,13.0,205.19
3,335982,CL White,6,10,0.0,0.0,60.00
4,335982,DJ Hussey,12,12,1.0,0.0,100.00
...,...,...,...,...,...,...,...
16510,1426312,SP Narine,6,2,0.0,1.0,300.00
16511,1426312,SS Iyer,6,3,1.0,0.0,200.00
16512,1426312,Shahbaz Ahmed,8,7,0.0,1.0,114.29
16513,1426312,TM Head,0,1,0.0,0.0,0.00


In [66]:
final_df

,match_id,batter,runs,balls,fours,sixes,sr
0,335982,AA Noffke,9,12,1.0,0.0,75.00
1,335982,B Akhil,0,2,0.0,0.0,0.00
2,335982,BB McCullum,158,77,10.0,13.0,205.19
3,335982,CL White,6,10,0.0,0.0,60.00
4,335982,DJ Hussey,12,12,1.0,0.0,100.00
...,...,...,...,...,...,...,...
16510,1426312,SP Narine,6,2,0.0,1.0,300.00
16511,1426312,SS Iyer,6,3,1.0,0.0,200.00
16512,1426312,Shahbaz Ahmed,8,7,0.0,1.0,114.29
16513,1426312,TM Head,0,1,0.0,0.0,0.00


In [67]:
final_df = final_df.merge(temp_df, left_on=['match_id','batter'], right_on=['Match_Id','Player_Name'], how='left').drop(columns=['Player_Name', 'Match_Id']).fillna(0)

In [71]:
final_df.rename(columns={
    'match_id':'ID'
}, inplace=True)

In [72]:
final_df

,ID,batter,runs,balls,fours,sixes,sr,Is_Captain
0,335982,AA Noffke,9,12,1.0,0.0,75.00,0.0
1,335982,B Akhil,0,2,0.0,0.0,0.00,0.0
2,335982,BB McCullum,158,77,10.0,13.0,205.19,0.0
3,335982,CL White,6,10,0.0,0.0,60.00,0.0
4,335982,DJ Hussey,12,12,1.0,0.0,100.00,0.0
...,...,...,...,...,...,...,...,...
16510,1426312,SP Narine,6,2,0.0,1.0,300.00,0.0
16511,1426312,SS Iyer,6,3,1.0,0.0,200.00,0.0
16512,1426312,Shahbaz Ahmed,8,7,0.0,1.0,114.29,0.0
16513,1426312,TM Head,0,1,0.0,0.0,0.00,0.0


In [85]:
def dream11(row):
    
    score = 0
    
    score = score + row['runs'] + row['fours'] + 2*row['sixes']
    
    if row['runs'] >= 100:
        score = score + 16
        
    elif row['runs'] >=50 and row['runs'] < 100:
        score = score + 8
        
    elif row['runs'] >= 30 and row['runs'] < 50:
        score = score + 4
        
    elif row['runs'] == 0:
        score = score - 2
        
    
    if row['balls'] >= 10:
        
        if row['sr'] > 170:
            score = score + 6
        
        elif row['sr'] > 150 and row['sr'] <=170:
            score = score + 4
        
        elif row['sr'] > 130 and row['sr'] <=150:
            score = score + 2
        
        elif row['sr'] > 60 and row['sr'] <=70:
            score = score - 2
        
        elif row['sr'] > 50 and row['sr'] <=60:
            score = score - 4
        
        elif row['sr'] <= 50:
            score = score - 6
    
    
    if row['Is_Captain'] == 1:
        score = score * 2
    
    
    
    return score

In [87]:
final_df['score'] = final_df.apply(dream11, axis=1)

In [90]:
export_df = final_df.sort_values('score', ascending=False)[['ID', 'batter', 'score']]

In [91]:
export_df

,ID,batter,score
3325,501243,V Sehwag,332.0
2844,501210,SR Tendulkar,272.0
5302,598027,CH Gayle,244.0
2,335982,BB McCullum,216.0
4254,548342,V Sehwag,202.0
...,...,...,...
3546,501258,ND Doshi,-8.0
13132,1304051,N Pooran,-8.0
3648,501266,S Badrinath,-8.0
4048,548325,DJ Jacobs,-8.0


In [92]:
export_df.rename(columns={
    'ID':'match_id',
    'batter':'batsman_name',
    'score':'dream11_score',
}, inplace=True)

In [93]:
export_df

,match_id,batsman_name,dream11_score
3325,501243,V Sehwag,332.0
2844,501210,SR Tendulkar,272.0
5302,598027,CH Gayle,244.0
2,335982,BB McCullum,216.0
4254,548342,V Sehwag,202.0
...,...,...,...
3546,501258,ND Doshi,-8.0
13132,1304051,N Pooran,-8.0
3648,501266,S Badrinath,-8.0
4048,548325,DJ Jacobs,-8.0
